# 08 - Decoding strategies: greedy vs. beam search vs. beam search + n-gram LM

Loads the winning checkpoint from `notebooks/07_crnn_architecture_variants.ipynb`
(`models/crnn_residual_aug.pt`, regularized residual-block CRNN + data
augmentation, score 0.769) and compares three decode-time strategies on the
same fixed 120-row val sample - no retraining, this only changes how the
model's output probabilities are turned into text:

1. **Greedy decode** (argmax per time step) - the current default, 0.769.
2. **CTC prefix beam search**, no language model (Graves et al. 2006;
   algorithm per Hannun 2017, distill.pub - see RESOURCES.md). Already tried
   in notebook 07 on `crnn_residual_aug.pt`: `score = 0.721` at
   `beam_width=10` - within noise of greedy, logged as negative/inconclusive.
3. **CTC prefix beam search + a character n-gram language model** (shallow
   fusion, Amodei et al. 2016, "Deep Speech 2" - see RESOURCES.md), trained
   on this project's own fit-split `Target` text (add-k smoothed counting,
   not KenLM - avoids a native-dependency install for a corpus this small,
   ~200K characters). This is the one untested idea from today: beam search
   alone barely moved the needle because the CRNN's own outputs are already
   confident and there's no domain language model to disambiguate close
   calls - a same-corpus n-gram LM is meant to fix exactly that gap.

**If notebook 07's elastic-distortion continuation ended up beating 0.769,
update `CHECKPOINT_PATH` in the load-checkpoint cell below** to point at
that file before running the rest of this notebook.

In [ ]:
import sys
sys.path.append('..')

import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from src.config import RANDOM_STATE, MODELS_DIR
from src.data import build_image_path, load_raw_data, split_train_val
from src.evaluate import combined_score

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
train = load_raw_data('train')
fit_df, val_df = split_train_val(train)

# Same fixed sample/seed as notebooks 05 and 07, so scores stay comparable.
val_sample = val_df.sample(120, random_state=RANDOM_STATE).reset_index(drop=True)
len(fit_df), len(val_df), len(val_sample)

In [ ]:
# Character vocabulary from the fit split - deterministic given the same
# fit_df, so this matches what the checkpoint was trained with.
chars = sorted(set(''.join(fit_df['Target'].tolist())))
BLANK = 0
char_to_idx = {c: i + 1 for i, c in enumerate(chars)}
idx_to_char = {i + 1: c for i, c in enumerate(chars)}
NUM_CLASSES = len(chars) + 1
NUM_CLASSES

In [ ]:
TARGET_HEIGHT = 32
MAX_WIDTH = 1024


def load_and_resize(record_id):
    img = Image.open(build_image_path(record_id)).convert('L')
    w, h = img.size
    new_w = max(8, min(MAX_WIDTH, round(w * TARGET_HEIGHT / h)))
    img = img.resize((new_w, TARGET_HEIGHT), Image.BILINEAR)
    arr = (np.asarray(img, dtype=np.float32) / 255.0 - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0)  # 1 x H x W


def naive_output_len(w):
    """Output width after two MaxPool2d(2, 2) layers - matches CRNNResidual's
    downsampling (pooling unchanged from the plain CRNN baseline)."""
    return (w // 2) // 2


class LineDataset(Dataset):
    def __init__(self, df):
        self.ids = df['ID'].tolist()
        self.targets = df['Target'].tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img = load_and_resize(self.ids[i])
        text = self.targets[i]
        label = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
        return img, label, text


def collate_fn(batch):
    imgs, labels, texts = zip(*batch)
    widths = [im.shape[-1] for im in imgs]
    max_w = max(widths)
    padded = torch.zeros(len(imgs), 1, TARGET_HEIGHT, max_w)
    for i, im in enumerate(imgs):
        padded[i, :, :, : im.shape[-1]] = im
    input_lengths = torch.tensor([naive_output_len(w) for w in widths], dtype=torch.long)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    return padded, torch.cat(labels), input_lengths, target_lengths, texts


val_ds = LineDataset(val_sample)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
def greedy_decode(log_probs, input_lengths):
    """Argmax per time step, collapse repeats, drop blanks."""
    pred_idx = log_probs.argmax(2)
    texts = []
    for b in range(pred_idx.size(0)):
        seq = pred_idx[b, : input_lengths[b]].tolist()
        out, prev = [], None
        for idx in seq:
            if idx != prev and idx != BLANK:
                out.append(idx_to_char[idx])
            prev = idx
        texts.append(''.join(out))
    return texts

## Load the winning architecture and checkpoint

In [ ]:
class ResidualBlock(nn.Module):
    """Basic residual block (He et al. 2015 - see RESOURCES.md). Identical
    to notebooks/07_crnn_architecture_variants.ipynb."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_channels, out_channels, 1), nn.BatchNorm2d(out_channels))
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class CRNNResidual(nn.Module):
    """Residual-block CRNN (He et al. 2015), identical to
    notebooks/07_crnn_architecture_variants.ipynb - the architecture behind
    the current reference model, crnn_residual_aug.pt (0.769)."""

    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            ResidualBlock(1, 32), nn.MaxPool2d(2, 2),
            ResidualBlock(32, 64), nn.MaxPool2d(2, 2),
            ResidualBlock(64, 128), nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),
            ResidualBlock(128, 128), nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),
            nn.Conv2d(128, 256, (2, 1), 1, 0), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.rnn = nn.LSTM(256, 128, num_layers=2, bidirectional=True, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        feat = self.cnn(x)
        feat = feat.squeeze(2).permute(0, 2, 1)
        seq, _ = self.rnn(feat)
        seq = self.dropout(seq)
        return self.fc(seq).log_softmax(2)

In [ ]:
# Update this path if notebook 07's elastic-distortion continuation beat 0.769.
CHECKPOINT_PATH = MODELS_DIR / 'crnn_residual_aug.pt'

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model = CRNNResidual(NUM_CLASSES, dropout=0.2).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

results = {}

refs, hyps_greedy = [], []
with torch.no_grad():
    for imgs, targets_cat, input_lengths, target_lengths, texts in val_dl:
        imgs = imgs.to(device)
        log_probs = model(imgs).cpu()
        hyps_greedy.extend(greedy_decode(log_probs, input_lengths))
        refs.extend(texts)

results['greedy'] = combined_score(refs, hyps_greedy)
print(f"greedy (loaded checkpoint): score {results['greedy']['score']:.3f} "
      f"(expected ~0.769 if CHECKPOINT_PATH is crnn_residual_aug.pt)")

## Beam search decoder (with optional LM shallow fusion)

In [ ]:
NEG_INF = float('-inf')


def _logsumexp(*args):
    m = max(args)
    if m == NEG_INF:
        return NEG_INF
    return m + math.log(sum(math.exp(a - m) for a in args))


class CharNgramLM:
    """Character n-gram LM, add-k (Laplace) smoothed counting, trained on
    this project's own fit-split Target text - not KenLM, to avoid a C++
    native-dependency install for a corpus this small (~200K chars). Used
    for CTC beam search shallow fusion (Amodei et al. 2016 - see
    RESOURCES.md)."""

    def __init__(self, texts, n=3, k=1e-3):
        self.n = n
        self.k = k
        self.ngrams = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()
        for text in texts:
            padded = "^" * (n - 1) + text + "$"
            for i in range(len(padded) - n + 1):
                context = padded[i:i + n - 1]
                char = padded[i + n - 1]
                self.ngrams[context][char] += 1
                self.context_counts[context] += 1
                self.vocab.add(char)
        self.vocab_size = len(self.vocab)

    def get_log_prob(self, char, context):
        if len(context) > self.n - 1:
            context = context[-(self.n - 1):]
        elif len(context) < self.n - 1:
            context = "^" * (self.n - 1 - len(context)) + context
        total = self.context_counts[context]
        count = self.ngrams[context][char]
        prob = (count + self.k) / (total + self.k * self.vocab_size)
        return math.log(prob)


def beam_search_decode(log_probs, input_length, beam_width=10, top_k=25, lm=None, alpha=0.0):
    """CTC prefix beam search, optionally with character-LM shallow fusion
    (Graves et al. 2006; Amodei et al. 2016 for the fusion formula;
    algorithm per Hannun 2017 - see RESOURCES.md). Tracks p_b/p_nb (blank-
    vs non-blank-ending paths) separately so repeated characters collapse
    per the real CTC rule, and merges duplicate prefixes via logsumexp
    (sums probability mass over every alignment), not max - verified
    against a brute-force CTC forward score. LM score is added once per
    newly emitted symbol (not per timestep):
    score = log P_ctc + alpha * log P_lm."""
    log_probs = log_probs[:input_length]
    T, C = log_probs.shape
    k = min(top_k, C)
    beams = {(): (0.0, NEG_INF)}

    for t in range(T):
        topk_logp, topk_idx = log_probs[t].topk(k)
        next_beams = {}

        for prefix, (p_b, p_nb) in beams.items():
            p_total = _logsumexp(p_b, p_nb)
            end_char = prefix[-1] if prefix else None
            prefix_str = ''.join(idx_to_char[x] for x in prefix)

            for log_p, c in zip(topk_logp.tolist(), topk_idx.tolist()):
                if c == BLANK:
                    nb = next_beams.get(prefix, (NEG_INF, NEG_INF))
                    next_beams[prefix] = (_logsumexp(nb[0], p_total + log_p), nb[1])
                elif c == end_char:
                    nb = next_beams.get(prefix, (NEG_INF, NEG_INF))
                    next_beams[prefix] = (nb[0], _logsumexp(nb[1], p_nb + log_p))
                    lm_score = alpha * lm.get_log_prob(idx_to_char[c], prefix_str) if lm else 0.0
                    new_prefix = prefix + (c,)
                    nb2 = next_beams.get(new_prefix, (NEG_INF, NEG_INF))
                    next_beams[new_prefix] = (nb2[0], _logsumexp(nb2[1], p_b + log_p + lm_score))
                else:
                    lm_score = alpha * lm.get_log_prob(idx_to_char[c], prefix_str) if lm else 0.0
                    new_prefix = prefix + (c,)
                    nb2 = next_beams.get(new_prefix, (NEG_INF, NEG_INF))
                    next_beams[new_prefix] = (nb2[0], _logsumexp(nb2[1], p_total + log_p + lm_score))

        beams = dict(sorted(next_beams.items(), key=lambda kv: _logsumexp(*kv[1]), reverse=True)[:beam_width])

    best_prefix = max(beams.items(), key=lambda kv: _logsumexp(*kv[1]))[0]
    return ''.join(idx_to_char[c] for c in best_prefix)


def beam_decode_batch(log_probs, input_lengths, beam_width=10, lm=None, alpha=0.0):
    return [beam_search_decode(log_probs[b], input_lengths[b], beam_width=beam_width, lm=lm, alpha=alpha)
            for b in range(log_probs.size(0))]

**Smoke test (CPU, tiny synthetic input, no model/data needed).** Confirms
the repeat-collapse rule and logsumexp merging work (regression check versus
an earlier, buggy pasted implementation that produced one character per
timestep - see RESOURCES.md/README.md), and that turning the LM on actually
changes the decoded output on a toy example, before running either on real
data.

In [ ]:
_toy_idx_to_char = {1: 'a', 2: 'b'}
_orig_idx_to_char = idx_to_char
idx_to_char = _toy_idx_to_char  # temporarily swap in the toy vocab for this smoke test
BLANK = 0

# Deterministic case with a known-by-construction answer: near-one-hot logits
# force the frame-by-frame argmax path to be exactly a, a, blank, a. Correctly
# collapsed CTC output: a,a (no blank between - collapses to one 'a'), then
# blank, then a (separated by the blank - counts again) = "aa".
HOT, COLD = 20.0, 0.0
path = [1, 1, BLANK, 1]  # a, a, blank, a
logits = torch.full((4, 3), COLD)
for t, c in enumerate(path):
    logits[t, c] = HOT
log_probs_det = logits.log_softmax(-1)

out_det = beam_search_decode(log_probs_det, 4, beam_width=5, top_k=3)
assert out_det == "aa", f"repeat-collapse regression: expected 'aa', got {out_det!r}"
print(f"repeat-collapse check: path a,a,blank,a -> {out_det!r} (expected 'aa') OK")

lm_toy = CharNgramLM(["aabaabaabaab"] * 20 + ["abbabbabbabb"], n=3, k=1e-3)
log_probs_toy = torch.log(torch.tensor([
    [0.05, 0.5, 0.45],
    [0.05, 0.45, 0.5],
    [0.05, 0.48, 0.47],
]))
out_no_lm = beam_search_decode(log_probs_toy, 3, beam_width=10, top_k=3, lm=None, alpha=0.0)
out_with_lm = beam_search_decode(log_probs_toy, 3, beam_width=10, top_k=3, lm=lm_toy, alpha=2.0)
assert out_no_lm != out_with_lm, "LM had no effect on a case designed to trigger it"
print(f"LM changes the decision: no LM -> {out_no_lm!r}, with LM -> {out_with_lm!r} (OK)")

idx_to_char = _orig_idx_to_char  # restore the real vocab

## Beam search alone (no LM) - sanity check against notebook 07's result

In [ ]:
refs, hyps_beam = [], []
with torch.no_grad():
    for imgs, targets_cat, input_lengths, target_lengths, texts in val_dl:
        imgs = imgs.to(device)
        log_probs = model(imgs).cpu()
        hyps_beam.extend(beam_decode_batch(log_probs, input_lengths, beam_width=10))
        refs.extend(texts)

results['beam_no_lm'] = combined_score(refs, hyps_beam)
print(f"beam search, no LM: score {results['beam_no_lm']['score']:.3f} "
      f"(greedy was {results['greedy']['score']:.3f}, "
      f"delta {results['beam_no_lm']['score'] - results['greedy']['score']:+.3f})")

## Beam search + character n-gram LM: sweep the fusion weight `alpha`

In [ ]:
lm = CharNgramLM(fit_df['Target'].tolist(), n=4, k=1e-3)

for alpha in [0.0, 0.3, 0.6, 1.0, 1.5, 2.0]:
    refs, hyps = [], []
    with torch.no_grad():
        for imgs, targets_cat, input_lengths, target_lengths, texts in val_dl:
            imgs = imgs.to(device)
            log_probs = model(imgs).cpu()
            hyps.extend(beam_decode_batch(log_probs, input_lengths, beam_width=10, lm=lm, alpha=alpha))
            refs.extend(texts)
    score = combined_score(refs, hyps)
    results[f'beam_lm_alpha_{alpha}'] = score
    print(f"alpha={alpha:.1f} | score {score['score']:.3f} "
          f"(greedy {results['greedy']['score']:.3f}, delta {score['score'] - results['greedy']['score']:+.3f})")

## Summary

Compare every row against `greedy` (the current reference, 0.769). If the
best `alpha` beats it - not just numerically but also on a qualitative
spot-check (same style as notebooks 05/07's 8-line check) - it becomes the
default decoding strategy for this checkpoint; log the result in
`README.md` either way (win or negative/inconclusive, per this project's
gate).

In [ ]:
summary = pd.DataFrame([
    {'strategy': name, **score} for name, score in results.items()
]).sort_values('score', ascending=False)
summary